In [7]:
from dotenv import load_dotenv

load_dotenv()

True

# 1.State 短期记忆

## 1.1自定义State

In [8]:
# State 短期记忆
from langchain.agents import AgentState
from typing import NotRequired


# 定义自定义State结构
class CustomState(AgentState):
    """Agent的任务状态"""
    model_call_count: NotRequired[int]  # 模型调用次数
    session_start: NotRequired[str]  # 会话开始时间

## 1.2 在工具中访问state

In [9]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
from datetime import datetime


@tool
def update_state(runtime: ToolRuntime):
    """A tool that update agent state"""
    # 获取state中的历史消息
    messages = runtime.state['messages']
    # 消息数量
    message_count = len(messages)
    # 组织结果
    command = {
        "model_call_count": runtime.state.get("model_call_count", 0) + 1,
        "messages": [ToolMessage("Successfully updated agent state", tool_call_id=runtime.tool_call_id)]
    }
    # 判断是否是第一次
    if message_count <= 2:
        command['session_start'] = datetime.now()

    return Command(update=command)

## 1.3 设置state schema

In [10]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "deepseek-v4-pro",
    tools=[update_state],
    state_schema=CustomState, # 告诉Agent使用我们自定义的state
    checkpointer=InMemorySaver(),
    system_prompt="你是一个热心的助手，你需要在每次请求时调用update_state工具以更新任务状态。"
)

## 1.4 测试

In [13]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
from datetime import datetime
from langchain.agents import AgentState
from typing import NotRequired
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

# 定义自定义State结构
class CustomState(AgentState):
    """Agent的任务状态"""
    model_call_count: NotRequired[int]  # 模型调用次数
    session_start: NotRequired[str]  # 会话开始时间

@tool
def update_state(runtime: ToolRuntime):
    """A tool that update agent state"""
    # 获取state中的历史消息
    messages = runtime.state['messages']
    # 消息数量
    message_count = len(messages)
    # 组织结果
    command = {
        "model_call_count": runtime.state.get("model_call_count", 0) + 1,
        "messages": [ToolMessage("Successfully updated agent state", tool_call_id=runtime.tool_call_id)]
    }
    # 判断是否是第一次
    if message_count <= 2:
        command['session_start'] = datetime.now()

    return Command(update=command)

agent = create_agent(
    "deepseek-chat",
    tools=[update_state],
    state_schema=CustomState,
    checkpointer=InMemorySaver(),
    system_prompt="你是一个热心的助手，你需要在每次请求时调用update_state工具以更新任务状态。"
)

config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [HumanMessage(content="Hi, my name is 虎哥")]},
    config
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Hi, my name is 虎哥
================================== Ai Message ==================================

你好！虎哥，很高兴认识你！😊

让我先更新一下任务状态。
Tool Calls:
  update_state (call_00_BEbAs9HsGuml0R5Mi1Co9826)
 Call ID: call_00_BEbAs9HsGuml0R5Mi1Co9826
  Args:
================================= Tool Message =================================
Name: update_state

Successfully updated agent state
================================== Ai Message ==================================

状态已更新！虎哥，有什么我可以帮你忙的吗？无论是日常问题、学习工作上的困惑，还是只是想聊聊天，我都很乐意陪你！🎉


In [14]:
agent.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi, my name is 虎哥', additional_kwargs={}, response_metadata={}, id='b7c57608-4ef6-4696-b3d7-26eae6800506'), AIMessage(content='你好！虎哥，很高兴认识你！😊\n\n让我先更新一下任务状态。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 285, 'total_tokens': 331, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 29}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '39826fb7-6330-4401-9cd8-88027de6d2bc', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f120b-fca8-70d1-9180-ecf71be55a00-0', tool_calls=[{'name': 'update_state', 'args': {}, 'id': 'call_00_BEbAs9HsGuml0R5Mi1Co9826', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 285, 'output_tokens'

# 2. Store（长期记忆）

store是LangChain提供的长期记忆机制，用于在不同会话间共享数据。例如：模型以外的数据、用户偏好等。
LangChain提供了多种Store的实现方式，例如：
- InMemoryStore
- PostgresStore
- RedisStore
课程中我们以InMemoryStore为例来学习。

## 2.1 Store的数据结构

In [15]:
from langgraph.store.memory import InMemoryStore

# ==================== 1. 创建Store ====================
memory_store = InMemoryStore()

In [16]:

# ==================== 2.初始化一些数据 ====================
memory_store.put(
    ("preferences",),  # namespace，是一个tuple
    "user_001",  # key，可以是任意类型
    {  # value，是JSON格式文档
        "style": "business_markdown",
        "language": "zh-CN"
    }
)

memory_store.put(("preferences",), "user_002", {
    "style": "trump",
    "language": "en-US"
})

In [17]:
# ==================== 3.读取 ====================

# 3.1.基于get查询
user_preferences = memory_store.get(("preferences",), "user_001")
print(f"用户信息: {user_preferences.value if user_preferences else 'Not found'}")

# 3.2.基于search搜索数据
search_results = memory_store.search(
    ("preferences",),
    filter={"language": "zh-CN"},  # 基于字段做过滤查询
    limit=5
)
print(f"搜索结果数量: {len(search_results)}")
print(search_results)

用户信息: {'style': 'business_markdown', 'language': 'zh-CN'}
搜索结果数量: 1
[Item(namespace=['preferences'], key='user_001', value={'style': 'business_markdown', 'language': 'zh-CN'}, created_at='2026-06-29T06:26:51.435754+00:00', updated_at='2026-06-29T06:26:51.435755+00:00', score=None)]


## 2.2 基于向量模型的Store

### 2.2.1 向量相似度

我们可以通过比较文字的向量相似度来判断词的含义是否接近。而向量相似度通常是基于向量之间的距离来计算的。

通常，两个向量之间距离越近，我们认为两个向量的相似度越高

所以，如果我们能把文本转为向量，就可以通过向量距离来判断文本的相似度了。

### 2.2.2 向量模型

现在，有不少的专门的向量模型，就可以实现将文本向量化。一个好的向量模型，就是要尽可能让文本含义相似的向量，在空间中距离更近,阿里云百炼就有这样的模型，最新模型名为text-embedding-v4

In [18]:
from langgraph_cli.schemas import IndexConfig
from langchain_community.embeddings import DashScopeEmbeddings
import os

# 初始化向量模型
embedding_model = DashScopeEmbeddings(
    model="text-embedding-v4", dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 初始化store
memory_store = InMemoryStore(index=IndexConfig(
    embed=embedding_model,  # 向量模型
    dims=1024  # 向量维度
))

store存储方式与之前没有区别

In [19]:
# ==================== 2.初始化一些数据 ====================
memory_store.put(("users",), "user_001", {
    "id": "user_001",
    "name": "张三",
    "department": "技术部",
    "clearance_level": 3
})

memory_store.put(("users",), "user_002", {
    "id": "user_002",
    "name": "李四",
    "department": "市场部",
    "clearance_level": 1
})

读取store时可以选择基于向量相似度的搜索方式

In [20]:
# ==================== 3.读取 ====================

# 3.1.基于get查询
user_data = memory_store.get(("users",), "user_001")
print(f"用户信息: {user_data.value if user_data else 'Not found'}")

# 3.2.基于search搜索数据
search_results = memory_store.search(
    ("users",),
    query="001",  # 基于字段语义检索
    limit=5
)
print(f"搜索结果数量: {len(search_results)}")
print(search_results)

用户信息: {'id': 'user_001', 'name': '张三', 'department': '技术部', 'clearance_level': 3}
搜索结果数量: 2
[Item(namespace=['users'], key='user_001', value={'id': 'user_001', 'name': '张三', 'department': '技术部', 'clearance_level': 3}, created_at='2026-06-29T06:35:36.704436+00:00', updated_at='2026-06-29T06:35:36.704439+00:00', score=0.3413479702327375), Item(namespace=['users'], key='user_002', value={'id': 'user_002', 'name': '李四', 'department': '市场部', 'clearance_level': 1}, created_at='2026-06-29T06:35:37.004723+00:00', updated_at='2026-06-29T06:35:37.004725+00:00', score=0.2891264007369341)]


可以看到搜索结果会有多个，并且每个都有一个score字段，也就是相似度得分，得分越高排名越靠前。

## 2.3 在tool中访问store

与state类似，LangChain中访问store通常也可以有两种场景：
- Tool
- 中间件
我们依然先学习tool中访问store，与state类似，在tool 中访问store也是通过runtime

依然使用上一小节的store为例，在tool中访问store要这样做：

In [21]:
from langchain.tools import tool, ToolRuntime

@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """获取用户信息"""
    if runtime.store is None:
        return "Store not available"

    # 通过runtime获取store，读取其中的数据
    user_info = runtime.store.get(("users",), user_id)

    if user_info is None:
        return "没有找到用户"

    return f"用户信息: {user_info.value}"

## 2.4 给agent添加store

In [22]:
from langchain.messages import HumanMessage

# 在Agent中集成
agent = create_agent(
    model="deepseek-chat",
    tools=[get_user_info],
    store=memory_store  # 指定store的存储方式
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询user_001的信息")]
})
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查询user_001的信息
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_00_APhU4e81lNmOMDmVRi7Z0346)
 Call ID: call_00_APhU4e81lNmOMDmVRi7Z0346
  Args:
    user_id: user_001
================================= Tool Message =================================
Name: get_user_info

用户信息: {'id': 'user_001', 'name': '张三', 'department': '技术部', 'clearance_level': 3}
================================== Ai Message ==================================

以下是用户 **user_001** 的信息：

| 字段 | 内容 |
|------|------|
| **用户ID** | user_001 |
| **姓名** | 张三 |
| **部门** | 技术部 |
| **权限等级** | 3 |

如果您需要进一步的信息，请随时告诉我！


# Context（运行上下文）

Context用于在运行时传递配置参数、用户信息等会话临时数据，你可以自定义Context的数据结构。

例如，在系统中用户通常会先登录，然后访问agent，我们就可以把登录用户信息存入Context，方便后续获取用户信息。

## 3.1 定义Context Schema

首先，我们需要约定Context的数据结构，有两种方式：
- 方式一：使用@dataclass来定义
- 方式二：继承TypedDict

In [23]:
from dataclasses import dataclass


# 方式1：使用dataclass定义Context
@dataclass
class UserContext:
    """Agent运行时上下文"""
    user_id: str = ""


# 方式2：使用TypedDict定义Context
from typing_extensions import TypedDict


class UserContext2(TypedDict):
    """运行时上下文类型"""
    user_id: str

## 3.2 在Tool中使用Context

In [24]:
from langchain.agents import create_agent

@tool
def get_users(runtime: ToolRuntime[UserContext]):
    """查询所有用户信息"""
    # 获取store
    store = runtime.store
    if store is None:
        return "Store not available"
    # 获取当前用户信息
    user_id = runtime.context.user_id
    if user_id is None:
        return "当前用户未登录，无法查看"

    user = store.get(("users",), user_id)
    if user is None:
        return "当前用户未登录，无法查看"

    # 校验权限，至少是3级权限
    user_info = dict(user.value)
    if user_info['clearance_level'] < 3:
        return "权限不足！"
    # 查询用户
    results = runtime.store.search(("users",))
    if results is None or len(results) == 0:
        return "未查询到用户"
    users = [item.value for item in results]
    return users


@tool
def get_user_preferences(runtime: ToolRuntime[UserContext]):
    """查询当前用户的偏好，根据偏好输出结果"""
    # 获取当前用户id
    user_id = runtime.context.user_id
    # 获取用户偏好
    user_preference = runtime.store.get(("preferences",), user_id)
    if user_preference is None:
        return "未查找到用户偏好信息"

    return user_preference.value

## 3.3 给Agent添加Context

In [25]:
# 创建Agent时指定context_schema
agent = create_agent(
    model="deepseek-chat",
    tools=[get_users, get_user_preferences],
    store=memory_store,
    context_schema=UserContext,  # 指定Context类型
    system_prompt="""
    # indentify
    你是一个热心的助手，你可以调用工具获取用户信息，用户偏好。
    # instruction
    请务必按照用户偏好风格展示结果。
    """
)

调用Agent时，可以传递Context信息：

In [26]:
# 调用时通过Context传递用户信息
response = agent.invoke(
    {"messages": [HumanMessage("Hello, 帮我查询所有用户信息")]},
    context=UserContext(user_id="user_001")
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Hello, 帮我查询所有用户信息
================================== Ai Message ==================================

好的，我来查询用户信息和当前用户的偏好。
Tool Calls:
  get_users (call_00_Put8O4dHxdXTDZrssEno9533)
 Call ID: call_00_Put8O4dHxdXTDZrssEno9533
  Args:
  get_user_preferences (call_01_04VNgHDAZdO7ZLHWfcUA1556)
 Call ID: call_01_04VNgHDAZdO7ZLHWfcUA1556
  Args:
================================= Tool Message =================================
Name: get_users

[{"id": "user_001", "name": "张三", "department": "技术部", "clearance_level": 3}, {"id": "user_002", "name": "李四", "department": "市场部", "clearance_level": 1}]
================================= Tool Message =================================
Name: get_user_preferences

未查找到用户偏好信息
================================== Ai Message ==================================

好的，以下是所有用户的信息：

---

### 📋 用户信息列表

| 编号 | 姓名 | 部门 | 权限等级 |
|:---:|:----:|:----:|:--------:|
| 🆔 user_001 | **张三** 🧑‍💻 | 技术部

# 4. 总结
1. State（短期记忆）
  - 通过runtime.state参数访问
  - 用于保存会话历史消息、对话状态、计数等临时信息
  - 生命周期是当前会话
  - 存储方式：可以是InMemorySaver、也可以是数据库
2. Store（长期记忆）
  - 通过runtime.store访问
  - 用于拓展知识、用户偏好等信息
  - 生命周期跨越多个会话
  - 支持InMemoryStore和数据库存储
3. Context（运行时上下文）
  - 通过runtime.context访问
  - 用于传递用户ID、配置参数等
  - 生命周期是当前会话
  - 存储方式：基于内存